# Notebook 02 — Data Preprocessing


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "online_food_delivery_analysis_dataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = PROCESSED_DIR / "online_food_delivery_analysis_cleaned.csv"

print("Project root:", PROJECT_ROOT.resolve())
print("Raw path:", RAW_PATH.resolve())
print("Raw file exists:", RAW_PATH.exists())

if not RAW_PATH.exists():
    raise FileNotFoundError(
        "Could not find the raw CSV. Put it at data/raw/"
        "online_food_delivery_analysis_dataset.csv, or update RAW_PATH."
    )

df_raw = pd.read_csv(RAW_PATH)
print("Raw dataset shape:", df_raw.shape)
display(df_raw.head())


Project root: D:\online food delivery analysis final
Raw path: D:\online food delivery analysis final\data\raw\online_food_delivery_analysis_dataset.csv
Raw file exists: True
Raw dataset shape: (100000, 25)


,Order_ID,Customer_ID,Customer_Age,Customer_Gender,City,Area,Restaurant_ID,Restaurant_Name,Cuisine_Type,Order_Date,...,Final_Amount,Payment_Mode,Order_Status,Cancellation_Reason,Delivery_Partner_ID,Delivery_Rating,Restaurant_Rating,Order_Day,Peak_Hour,Profit_Margin
0,ORD000001,CUST03987,24.0,Male,Chennai,OMR,RES1298,Restaurant_1298,Fast Food,06/05/2024,...,1801.20,COD,Delivered,Not Applicable,DP00866,4.4,3.2,Weekday,False,0.029
1,ORD000002,CUST01917,45.0,Female,Hyderabad,NaN,RES1191,Restaurant_1191,Chinese,05/11/2024,...,469.78,Credit Card,Delivered,Not Applicable,DP04095,4.6,4.3,Weekend,False,0.214
2,ORD000003,CUST13338,57.0,Male,Bengaluru,Indiranagar,RES0681,Restaurant_0681,Indian,02/02/2024,...,468.45,UPI,Delivered,Not Applicable,DP01853,4.6,3.5,Weekday,False,0.204
3,ORD000004,CUST09787,66.0,Female,Chennai,Velachery,RES0453,Restaurant_0453,Indian,03/27/2024,...,681.18,UPI,Delivered,Not Applicable,DP04885,5.0,4.3,Weekday,True,0.293
4,ORD000005,CUST15793,43.0,Female,Chennai,Anna Nagar,RES0268,Restaurant_0268,Indian,07/11/2024,...,471.87,UPI,Preparing,Not Applicable,DP04754,NaN,4.7,Weekday,True,NaN


## 1. Validate the input and take an untouched snapshot


In [2]:
required_columns = [
    "Order_ID", "Customer_ID", "Customer_Age", "Customer_Gender",
    "City", "Area", "Restaurant_ID", "Restaurant_Name", "Cuisine_Type",
    "Order_Date", "Order_Time", "Delivery_Time_Min", "Distance_km",
    "Order_Value", "Discount_Applied", "Final_Amount", "Payment_Mode",
    "Order_Status", "Cancellation_Reason", "Delivery_Partner_ID",
    "Delivery_Rating", "Restaurant_Rating", "Order_Day", "Peak_Hour",
    "Profit_Margin"
]

missing_required_columns = sorted(set(required_columns) - set(df_raw.columns))
if missing_required_columns:
    raise ValueError(f"Required columns missing from CSV: {missing_required_columns}")

raw_snapshot = df_raw.copy(deep=True)
raw_shape = df_raw.shape
raw_order_id_duplicates = int(df_raw["Order_ID"].duplicated().sum())
raw_exact_duplicates = int(df_raw.duplicated().sum())

print("Rows/columns:", raw_shape)
print("Exact duplicate rows in source:", raw_exact_duplicates)
print("Duplicate Order_ID values in source:", raw_order_id_duplicates)


Rows/columns: (100000, 25)
Exact duplicate rows in source: 0
Duplicate Order_ID values in source: 0


## 2. Create the working copy and standardize whitespace


In [3]:
df_clean = df_raw.copy(deep=True)

# Standardize whitespace around column names.
df_clean.columns = df_clean.columns.astype(str).str.strip()

# Strip whitespace in text values without changing missing values.
text_columns = df_clean.select_dtypes(include=["object", "string"]).columns
for column in text_columns:
    df_clean[column] = df_clean[column].map(
        lambda value: value.strip() if isinstance(value, str) else value
    )

print("Working copy shape:", df_clean.shape)
print("Raw equals snapshot after copy:", df_raw.equals(raw_snapshot))


Working copy shape: (100000, 25)
Raw equals snapshot after copy: True


## 3. Convert data types carefully


In [4]:
# Track non-empty values that fail conversion.
conversion_issues = {}

if "Order_Date" in df_clean.columns:
    original = df_clean["Order_Date"].copy()
    converted = pd.to_datetime(original, errors="coerce")
    conversion_issues["Order_Date newly unparseable"] = int(
        (original.notna() & converted.isna()).sum()
    )
    df_clean["Order_Date"] = converted

if "Order_Time" in df_clean.columns:
    original = df_clean["Order_Time"].copy()
    converted = pd.to_datetime(
        original.astype("string").str.strip(),
        format="%H:%M",
        errors="coerce"
    )
    conversion_issues["Order_Time newly unparseable"] = int(
        (original.notna() & converted.isna()).sum()
    )
    df_clean["Order_Time"] = converted.dt.time

numeric_columns = [
    "Customer_Age", "Delivery_Time_Min", "Distance_km",
    "Order_Value", "Discount_Applied", "Final_Amount",
    "Delivery_Rating", "Restaurant_Rating", "Profit_Margin"
]

for column in numeric_columns:
    original = df_clean[column].copy()
    converted = pd.to_numeric(original, errors="coerce")
    conversion_issues[f"{column} newly nonnumeric"] = int(
        (original.notna() & converted.isna()).sum()
    )
    df_clean[column] = converted

print("New conversion issues (ideally zero):")
display(pd.Series(conversion_issues, name="Count").to_frame())
print("\nData types after conversion:")
display(df_clean.dtypes.rename("Data_Type").to_frame())


New conversion issues (ideally zero):


,Count
Order_Date newly unparseable,0
Order_Time newly unparseable,0
Customer_Age newly nonnumeric,0
Delivery_Time_Min newly nonnumeric,0
Distance_km newly nonnumeric,0
Order_Value newly nonnumeric,0
Discount_Applied newly nonnumeric,0
Final_Amount newly nonnumeric,0
Delivery_Rating newly nonnumeric,0
Restaurant_Rating newly nonnumeric,0



Data types after conversion:


,Data_Type
Order_ID,str
Customer_ID,str
Customer_Age,float64
Customer_Gender,str
City,str
Area,str
Restaurant_ID,str
Restaurant_Name,str
Cuisine_Type,str
Order_Date,datetime64[us]


## 4. Review duplicates

In [5]:
exact_duplicates_after_standardization = int(df_clean.duplicated().sum())
duplicate_order_ids_after_standardization = int(df_clean["Order_ID"].duplicated().sum())

print("Exact duplicate rows after standardization:", exact_duplicates_after_standardization)
print("Duplicate Order_ID values after standardization:", duplicate_order_ids_after_standardization)

if exact_duplicates_after_standardization > 0 or duplicate_order_ids_after_standardization > 0:
    print("Review duplicate records before proceeding. No rows have been deleted.")
else:
    print("No duplicate records identified. No rows removed.")


Exact duplicate rows after standardization: 0
Duplicate Order_ID values after standardization: 0
No duplicate records identified. No rows removed.


## 5. Review missing values

In [6]:
missing_summary = pd.DataFrame({
    "Missing_Count": df_clean.isna().sum(),
    "Missing_Percent": (df_clean.isna().mean() * 100).round(2)
})
missing_summary = missing_summary[missing_summary["Missing_Count"] > 0]
missing_summary = missing_summary.sort_values("Missing_Count", ascending=False)

print("Missing-value summary:")
display(missing_summary)

for column in ["Delivery_Time_Min", "Delivery_Rating", "Profit_Margin"]:
    if column in df_clean.columns:
        print(f"\n{column} missingness by Order_Status:")
        display(pd.crosstab(
            df_clean["Order_Status"],
            df_clean[column].isna(),
            rownames=["Order_Status"],
            colnames=["Is_Missing"]
        ))


Missing-value summary:


,Missing_Count,Missing_Percent
Delivery_Rating,14379,14.38
Profit_Margin,10712,10.71
Delivery_Time_Min,10712,10.71
Restaurant_Rating,2586,2.59
Customer_Age,1476,1.48
Distance_km,1334,1.33
Final_Amount,997,1.00
Customer_Gender,990,0.99
City,548,0.55
Area,500,0.50



Delivery_Time_Min missingness by Order_Status:


Is_Missing,False,True
Order_Status,,
Cancelled,0,6143
Delivered,89288,0
Out for Delivery,0,2541
Preparing,0,2028



Delivery_Rating missingness by Order_Status:


Is_Missing,False,True
Order_Status,,
Cancelled,0,6143
Delivered,85621,3667
Out for Delivery,0,2541
Preparing,0,2028



Profit_Margin missingness by Order_Status:


Is_Missing,False,True
Order_Status,,
Cancelled,0,6143
Delivered,89288,0
Out for Delivery,0,2541
Preparing,0,2028


## 6. Safe evidence-based corrections


In [7]:
# Fill only missing Final_Amount values where both source inputs are available.
# Preserve all existing non-missing Final_Amount values.
financial_inputs_available = df_clean["Order_Value"].notna() & df_clean["Discount_Applied"].notna()
final_amount_was_missing = df_clean["Final_Amount"].isna()
can_calculate_final = final_amount_was_missing & financial_inputs_available

df_clean.loc[can_calculate_final, "Final_Amount"] = (
    df_clean.loc[can_calculate_final, "Order_Value"]
    - df_clean.loc[can_calculate_final, "Discount_Applied"]
)

print("Missing Final_Amount values filled from available inputs:", int(can_calculate_final.sum()))
print("Final_Amount values still missing:", int(df_clean["Final_Amount"].isna().sum()))

# Check formula consistency only where all relevant values are present.
financial_complete = df_clean[["Order_Value", "Discount_Applied", "Final_Amount"]].notna().all(axis=1)
expected_final = df_clean["Order_Value"] - df_clean["Discount_Applied"]
formula_mismatch = financial_complete & ~np.isclose(
    df_clean["Final_Amount"], expected_final, rtol=0, atol=0.01
)

print("Rows with all financial inputs:", int(financial_complete.sum()))
print("Final_Amount formula mismatches (tolerance 0.01):", int(formula_mismatch.sum()))

if formula_mismatch.any():
    display(df_clean.loc[
        formula_mismatch,
        ["Order_ID", "Order_Value", "Discount_Applied", "Final_Amount"]
    ].head(20))


Missing Final_Amount values filled from available inputs: 0
Final_Amount values still missing: 997
Rows with all financial inputs: 99003
Final_Amount formula mismatches (tolerance 0.01): 0


## 7. Validate ranges and business rules


In [8]:
validation = {}

# Key fields
validation["Missing Order_ID"] = int(df_clean["Order_ID"].isna().sum())
validation["Duplicate Order_ID"] = int(df_clean["Order_ID"].duplicated().sum())
validation["Exact duplicate rows"] = int(df_clean.duplicated().sum())

# Numeric non-negativity checks
for column in ["Delivery_Time_Min", "Distance_km", "Order_Value", "Discount_Applied", "Final_Amount"]:
    validation[f"Negative values: {column}"] = int((df_clean[column] < 0).sum())

# Provisional range checks
validation["Customer_Age outside 18–100 (review assumption)"] = int(
    (df_clean["Customer_Age"].notna() & ~df_clean["Customer_Age"].between(18, 100)).sum()
)
for column in ["Delivery_Rating", "Restaurant_Rating"]:
    validation[f"{column} outside 1–5 (review assumption)"] = int(
        (df_clean[column].notna() & ~df_clean[column].between(1, 5)).sum()
    )

# Delivery status checks
delivered = df_clean["Order_Status"].eq("Delivered")
not_delivered = ~delivered
validation["Delivered orders missing delivery time"] = int(
    (delivered & df_clean["Delivery_Time_Min"].isna()).sum()
)
validation["Non-delivered orders with delivery time"] = int(
    (not_delivered & df_clean["Delivery_Time_Min"].notna()).sum()
)
validation["Non-delivered orders with delivery rating"] = int(
    (not_delivered & df_clean["Delivery_Rating"].notna()).sum()
)
validation["Non-delivered orders with profit margin"] = int(
    (not_delivered & df_clean["Profit_Margin"].notna()).sum()
)

# Final amount should not be negative.
validation["Negative Final_Amount"] = int((df_clean["Final_Amount"] < 0).sum())

display(pd.Series(validation, name="Count").to_frame())


,Count
Missing Order_ID,0
Duplicate Order_ID,0
Exact duplicate rows,0
Negative values: Delivery_Time_Min,0
Negative values: Distance_km,0
Negative values: Order_Value,0
Negative values: Discount_Applied,0
Negative values: Final_Amount,0
Customer_Age outside 18–100 (review assumption),0
Delivery_Rating outside 1–5 (review assumption),0


## 8. Final integrity checks


In [9]:
raw_unchanged = df_raw.equals(raw_snapshot)
same_row_count = len(df_clean) == len(df_raw)
same_source_columns = list(df_clean.columns) == list(df_raw.columns)

print("Raw DataFrame unchanged:", raw_unchanged)
print("Same row count as raw:", same_row_count)
print("Same source columns and order:", same_source_columns)
print("Raw shape:", df_raw.shape)
print("Cleaned shape:", df_clean.shape)

if not raw_unchanged:
    raise AssertionError("Raw DataFrame changed unexpectedly.")
if not same_row_count:
    raise AssertionError("Row count changed. Investigate before saving.")
if not same_source_columns:
    raise AssertionError("Source columns changed. Investigate before saving.")


Raw DataFrame unchanged: True
Same row count as raw: True
Same source columns and order: True
Raw shape: (100000, 25)
Cleaned shape: (100000, 25)


## 9. Save the cleaned dataset


In [10]:
df_clean.to_csv(OUTPUT_PATH, index=False)

print("Cleaned dataset saved to:", OUTPUT_PATH.resolve())
print("File exists:", OUTPUT_PATH.exists())
print("Output shape:", df_clean.shape)

# Reload the saved output and verify the saved row/column counts.
df_saved_check = pd.read_csv(OUTPUT_PATH)
print("Reloaded output shape:", df_saved_check.shape)
print("Saved output row count matches df_clean:", len(df_saved_check) == len(df_clean))
print("Saved output column count matches df_clean:", len(df_saved_check.columns) == len(df_clean.columns))


Cleaned dataset saved to: D:\online food delivery analysis final\data\processed\online_food_delivery_analysis_cleaned.csv
File exists: True
Output shape: (100000, 25)
Reloaded output shape: (100000, 25)
Saved output row count matches df_clean: True
Saved output column count matches df_clean: True
